In [0]:
#Check the Spark Environment
spark

In [0]:
from pyspark.sql import SparkSession

# The professional way to initialize Spark
spark = SparkSession.builder \
    .appName("My First Spark App") \
    .getOrCreate()

In [0]:
# Asking the cluster to return the current Spark version
spark.sql("SELECT version()").show(truncate=False)

+----------------------------------------------+
|version()                                     |
+----------------------------------------------+
|4.1.0 0000000000000000000000000000000000000000|
+----------------------------------------------+



In [0]:
# Forcing Spark to distribute work across 4 tasks
spark.range(10).repartition(4).count()

10

In [0]:
# 1. Create a raw Python list of data tuples
mock_data = [
    (1, "Project Alpha"), 
    (2, "Project Beta"), 
    (3, "Project Gamma")
]

# 2. Convert the Python list into a PySpark DataFrame
df_python = spark.createDataFrame(mock_data, ["id", "project_name"])

display(df_python)

id,project_name
1,Project Alpha
2,Project Beta
3,Project Gamma


In [0]:
# 3. Register the DataFrame in the Spark SQL Catalog
df_python.createOrReplaceTempView("project_codes")

In [0]:
%sql
-- 4. Query the Python data using pure SQL!
SELECT * FROM project_codes
WHERE id > 1

id,project_name
2,Project Beta
3,Project Gamma


In [0]:
# 5. Capture the SQL output back into a Python variable
final_python_df = _sqldf

print("Welcome back to Python!")
display(final_python_df)

Welcome back to Python!


id,project_name
2,Project Beta
3,Project Gamma


In [0]:
# 1. List the contents of our volume folder
folder_path = "/Volumes/workspace/default/course_data"

# Call the list command
files = dbutils.fs.ls(folder_path)

display(files)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/course_data/events.json,events.json,1434,1773086074000
dbfs:/Volumes/workspace/default/course_data/output/,output/,0,1773937304959
dbfs:/Volumes/workspace/default/course_data/sales_data.csv,sales_data.csv,805,1773086075000


In [0]:
# 2. Loop through the file list to find CSVs programmatically
for file_info in files:
    
    # Check if the name ends with .csv
    if file_info.name.endswith(".csv"):
        print(f"Found a CSV: {file_info.name} (Size: {file_info.size} bytes)")

Found a CSV: sales_data.csv (Size: 805 bytes)


In [0]:
# 3. Read the first 200 bytes of the file as plain text
csv_path = "/Volumes/workspace/default/course_data/sales_data.csv"

# Print the raw text
print(dbutils.fs.head(csv_path, 200))

[Truncated to first 200 bytes]
transaction_id,customer_id,product,amount,transaction_date,region
T1001,C001,Laptop,1200.50,2024-01-15,North
T1002,C002,Mouse,25.00,2024-01-15,South
T1003,C003,Monitor,350.00,2024-01-16,East
T1004


In [0]:
# The "Eager" Read (How beginners do it)
csv_path = "/Volumes/workspace/default/course_data/sales_data.csv"

# Tell Spark to read the file and guess the column types
df_naive = spark.read.csv(
    csv_path,
    header=True,
    inferSchema=True
)

In [0]:
# Triggering an Action
display(df_naive)

transaction_id,customer_id,product,amount,transaction_date,region
T1001,C001,Laptop,1200.5,2024-01-15,North
T1002,C002,Mouse,25.0,2024-01-15,South
T1003,C003,Monitor,350.0,2024-01-16,East
T1004,C001,Keyboard,45.0,2024-01-16,North
T1005,C004,Laptop,1150.0,2024-01-17,West
T1006,C002,Webcam,85.0,2024-01-17,South
T1007,C005,Headphones,120.0,2024-01-18,North
T1008,C003,Monitor,350.0,2024-01-18,East
T1009,C001,Docking Station,150.0,2024-01-19,North
T1010,C004,Mouse Pad,15.0,2024-01-19,West


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

# 1. Define the exact blueprint of our data
sales_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("transaction_date", DateType(), True),
    StructField("region", StringType(), True)
])

# 2. The "Lazy" Read (The Professional Way)
csv_path = "/Volumes/workspace/default/course_data/sales_data.csv"

df_fast = spark.read.schema(sales_schema).csv(csv_path, header=True)

In [0]:
# Trigger the actual read
display(df_fast)

transaction_id,customer_id,product,amount,transaction_date,region
T1001,C001,Laptop,1200.5,2024-01-15,North
T1002,C002,Mouse,25.0,2024-01-15,South
T1003,C003,Monitor,350.0,2024-01-16,East
T1004,C001,Keyboard,45.0,2024-01-16,North
T1005,C004,Laptop,1150.0,2024-01-17,West
T1006,C002,Webcam,85.0,2024-01-17,South
T1007,C005,Headphones,120.0,2024-01-18,North
T1008,C003,Monitor,350.0,2024-01-18,East
T1009,C001,Docking Station,150.0,2024-01-19,North
T1010,C004,Mouse Pad,15.0,2024-01-19,West


In [0]:
# 1. Define the path to our JSON file
json_path = "/Volumes/workspace/default/course_data/events.json"

# 2. Read the JSON into a DataFrame
events_df = spark.read.json(json_path)

# 3. Trigger an action to see the data
display(events_df)

details,device,event_id,event_type,timestamp,user_id
null,mobile,E001,LOGIN,2024-01-15T08:30:00,C001
"List(Electronics, null, null, null, Laptop, null, null, null)",null,E002,VIEW_PRODUCT,2024-01-15T08:35:00,C001
"List(Accessories, null, null, null, Mouse, null, null, null)",null,E003,VIEW_PRODUCT,2024-01-15T09:00:00,C002
"List(null, null, null, null, Mouse, 1, null, null)",null,E004,ADD_TO_CART,2024-01-15T09:05:00,C002
null,web,E005,LOGIN,2024-01-16T10:00:00,C003
"List(null, null, null, null, Monitor, null, 27-inch, null)",null,E006,VIEW_PRODUCT,2024-01-16T10:15:00,C003
"List(null, USD, null, null, null, null, null, 1200.5)",null,E007,PURCHASE,2024-01-16T11:20:00,C001
null,null,E008,LOGOUT,2024-01-16T12:00:00,C003
"List(null, null, 404, Page not found, null, null, null, null)",null,E009,ERROR,2024-01-17T14:30:00,C005
null,tablet,E010,LOGIN,2024-01-18T09:00:00,C004


In [0]:
# Print the inferred schema of the JSON file
events_df.printSchema()

root
 |-- details: struct (nullable = true)
 |    |-- category: string (nullable = true)
 |    |-- currency: string (nullable = true)
 |    |-- error_code: long (nullable = true)
 |    |-- message: string (nullable = true)
 |    |-- product: string (nullable = true)
 |    |-- quantity: long (nullable = true)
 |    |-- screen_size: string (nullable = true)
 |    |-- total_amount: double (nullable = true)
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- user_id: string (nullable = true)



In [0]:
from pyspark.sql.functions import col

# Select top-level and nested columns using Dot Notation
events_df.select(
    col("event_id"),
    col("event_type"),
    col("details.product").alias("product_name"),
    col("details.category").alias("product_category")
).display()

event_id,event_type,product_name,product_category
E001,LOGIN,null,null
E002,VIEW_PRODUCT,Laptop,Electronics
E003,VIEW_PRODUCT,Mouse,Accessories
E004,ADD_TO_CART,Mouse,null
E005,LOGIN,null,null
E006,VIEW_PRODUCT,Monitor,null
E007,PURCHASE,null,null
E008,LOGOUT,null,null
E009,ERROR,null,null
E010,LOGIN,null,null


In [0]:
# Safer traversal for column names containing dots or special characters
events_df.select(
    col("event_id"),
    col("details")["product"].alias("product_name")
).display()

event_id,product_name
E001,null
E002,Laptop
E003,Mouse
E004,Mouse
E005,null
E006,Monitor
E007,null
E008,null
E009,null
E010,null


In [0]:
# The Professional Flattening Move
flattened_df = events_df.select(
    "event_id",
    "event_type",
    "user_id",
    "timestamp",
    "details.*"  # <-- Expands all 8 sub-fields at once!
)

display(flattened_df)

event_id,event_type,user_id,timestamp,category,currency,error_code,message,product,quantity,screen_size,total_amount
E001,LOGIN,C001,2024-01-15T08:30:00,null,null,null,null,null,null,null,null
E002,VIEW_PRODUCT,C001,2024-01-15T08:35:00,Electronics,null,null,null,Laptop,null,null,null
E003,VIEW_PRODUCT,C002,2024-01-15T09:00:00,Accessories,null,null,null,Mouse,null,null,null
E004,ADD_TO_CART,C002,2024-01-15T09:05:00,null,null,null,null,Mouse,1,null,null
E005,LOGIN,C003,2024-01-16T10:00:00,null,null,null,null,null,null,null,null
E006,VIEW_PRODUCT,C003,2024-01-16T10:15:00,null,null,null,null,Monitor,null,27-inch,null
E007,PURCHASE,C001,2024-01-16T11:20:00,null,USD,null,null,null,null,null,1200.5
E008,LOGOUT,C003,2024-01-16T12:00:00,null,null,null,null,null,null,null,null
E009,ERROR,C005,2024-01-17T14:30:00,null,null,404,Page not found,null,null,null,null
E010,LOGIN,C004,2024-01-18T09:00:00,null,null,null,null,null,null,null,null


In [0]:
# 1. Create mock data to demonstrate array flattening
data = [
    ("O1", "User_A", ["Laptop", "Mouse"]), # Normal array
    ("O2", "User_B", []),                  # Empty array
    ("O3", "User_C", None)                 # Null array
]

# 2. Build the DataFrame
orders_df = spark.createDataFrame(data, ["order_id", "user_id", "line_items"])

display(orders_df)

order_id,user_id,line_items
O1,User_A,"List(Laptop, Mouse)"
O2,User_B,List()
O3,User_C,null


In [0]:
from pyspark.sql.functions import explode

# The Dangerous Way
exploded_df = orders_df.select(
    "order_id",
    "user_id",
    explode("line_items").alias("item")
)

display(exploded_df)

order_id,user_id,item
O1,User_A,Laptop
O1,User_A,Mouse


In [0]:
from pyspark.sql.functions import explode_outer

# The Safe Way
safe_exploded_df = orders_df.select(
    "order_id",
    "user_id",
    explode_outer("line_items").alias("item")
)

display(safe_exploded_df)

order_id,user_id,item
O1,User_A,Laptop
O1,User_A,Mouse
O2,User_B,null
O3,User_C,null


In [0]:
# Setup: Load the Sales Data with an Explicit Schema
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

sales_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("transaction_date", DateType(), True),
    StructField("region", StringType(), True)
])

path = "/Volumes/workspace/default/course_data/sales_data.csv"
sales_df = spark.read.schema(sales_schema).csv(path, header=True)

In [0]:
from pyspark.sql.functions import col, lit

# Creating a constant column alongside our real product column
sales_df.select(
    col("product"),
    lit(10).alias("flat_shipping_fee")
).display()

product,flat_shipping_fee
Laptop,10
Mouse,10
Monitor,10
Keyboard,10
Laptop,10
Webcam,10
Headphones,10
Monitor,10
Docking Station,10
Mouse Pad,10


In [0]:
from pyspark.sql.functions import col

sales_df.select(
    col("product"),
    col("amount"),
    (col("amount") * 1.15).alias("amount_with_tax")
).display()

product,amount,amount_with_tax
Laptop,1200.5,1380.5749999999998
Mouse,25.0,28.749999999999996
Monitor,350.0,402.49999999999994
Keyboard,45.0,51.74999999999999
Laptop,1150.0,1322.5
Webcam,85.0,97.74999999999999
Headphones,120.0,138.0
Monitor,350.0,402.49999999999994
Docking Station,150.0,172.5
Mouse Pad,15.0,17.25


In [0]:
# The Trap: DO NOT DO THIS IN LOOPS
df = sales_df.withColumn("tax", col("amount") * 0.1)
df = df.withColumn("discount", col("amount") * 0.2)

# Print the schema to analyze the added columns
df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- tax: double (nullable = true)
 |-- discount: double (nullable = true)



In [0]:
# The Professional Way: One plan, all at once.
sales_df.select(
    "*",  # Select all existing columns
    (col("amount") * 0.1).alias("tax"),
    (col("amount") * 0.2).alias("discount")
).display()

transaction_id,customer_id,product,amount,transaction_date,region,tax,discount
T1001,C001,Laptop,1200.5,2024-01-15,North,120.05000000000001,240.10000000000002
T1002,C002,Mouse,25.0,2024-01-15,South,2.5,5.0
T1003,C003,Monitor,350.0,2024-01-16,East,35.0,70.0
T1004,C001,Keyboard,45.0,2024-01-16,North,4.5,9.0
T1005,C004,Laptop,1150.0,2024-01-17,West,115.0,230.0
T1006,C002,Webcam,85.0,2024-01-17,South,8.5,17.0
T1007,C005,Headphones,120.0,2024-01-18,North,12.0,24.0
T1008,C003,Monitor,350.0,2024-01-18,East,35.0,70.0
T1009,C001,Docking Station,150.0,2024-01-19,North,15.0,30.0
T1010,C004,Mouse Pad,15.0,2024-01-19,West,1.5,3.0


In [0]:
# Filter by a simple condition
sales_df.filter(col("amount") > 500).display()

transaction_id,customer_id,product,amount,transaction_date,region
T1001,C001,Laptop,1200.5,2024-01-15,North
T1005,C004,Laptop,1150.0,2024-01-17,West
T1011,C002,Laptop,1200.5,2024-01-20,South
T1014,C003,Laptop,1150.0,2024-01-21,East


In [0]:
# Explicitly handling Nulls in Region
sales_df.filter(
    (col("region") != "North") | (col("region").isNull())
).display()

transaction_id,customer_id,product,amount,transaction_date,region
T1002,C002,Mouse,25.0,2024-01-15,South
T1003,C003,Monitor,350.0,2024-01-16,East
T1005,C004,Laptop,1150.0,2024-01-17,West
T1006,C002,Webcam,85.0,2024-01-17,South
T1008,C003,Monitor,350.0,2024-01-18,East
T1010,C004,Mouse Pad,15.0,2024-01-19,West
T1011,C002,Laptop,1200.5,2024-01-20,South
T1014,C003,Laptop,1150.0,2024-01-21,East
T1015,C004,Monitor,340.0,2024-01-22,West
T1017,C002,Mouse,25.0,2024-01-23,null


In [0]:
# The Correct Syntax with Bitwise Operators and Parentheses
sales_df.filter(
    (col("amount") > 100) & (col("region") == "North")
).display()


transaction_id,customer_id,product,amount,transaction_date,region
T1001,C001,Laptop,1200.5,2024-01-15,North
T1007,C005,Headphones,120.0,2024-01-18,North
T1009,C001,Docking Station,150.0,2024-01-19,North


In [0]:
from pyspark.sql.functions import col, lit

# Scenario A: Append a new column
sales_df.withColumn("source", lit("Marketing_Campaign")).display()

transaction_id,customer_id,product,amount,transaction_date,region,source
T1001,C001,Laptop,1200.5,2024-01-15,North,Marketing_Campaign
T1002,C002,Mouse,25.0,2024-01-15,South,Marketing_Campaign
T1003,C003,Monitor,350.0,2024-01-16,East,Marketing_Campaign
T1004,C001,Keyboard,45.0,2024-01-16,North,Marketing_Campaign
T1005,C004,Laptop,1150.0,2024-01-17,West,Marketing_Campaign
T1006,C002,Webcam,85.0,2024-01-17,South,Marketing_Campaign
T1007,C005,Headphones,120.0,2024-01-18,North,Marketing_Campaign
T1008,C003,Monitor,350.0,2024-01-18,East,Marketing_Campaign
T1009,C001,Docking Station,150.0,2024-01-19,North,Marketing_Campaign
T1010,C004,Mouse Pad,15.0,2024-01-19,West,Marketing_Campaign


In [0]:
# Scenario B: Replace 'amount' with an Integer version
# This "mutates" the schema and overrides the existing column
sales_df.withColumn("amount", col("amount").cast("integer")).display()

transaction_id,customer_id,product,amount,transaction_date,region
T1001,C001,Laptop,1200,2024-01-15,North
T1002,C002,Mouse,25,2024-01-15,South
T1003,C003,Monitor,350,2024-01-16,East
T1004,C001,Keyboard,45,2024-01-16,North
T1005,C004,Laptop,1150,2024-01-17,West
T1006,C002,Webcam,85,2024-01-17,South
T1007,C005,Headphones,120,2024-01-18,North
T1008,C003,Monitor,350,2024-01-18,East
T1009,C001,Docking Station,150,2024-01-19,North
T1010,C004,Mouse Pad,15,2024-01-19,West


In [0]:
from pyspark.sql.functions import concat

# The Dangerous Way: Standard concat()
sales_df.withColumn(
    "bad_description",
    concat(col("product"), lit(" - "), col("region"))
).display()

transaction_id,customer_id,product,amount,transaction_date,region,bad_description
T1001,C001,Laptop,1200.5,2024-01-15,North,Laptop - North
T1002,C002,Mouse,25.0,2024-01-15,South,Mouse - South
T1003,C003,Monitor,350.0,2024-01-16,East,Monitor - East
T1004,C001,Keyboard,45.0,2024-01-16,North,Keyboard - North
T1005,C004,Laptop,1150.0,2024-01-17,West,Laptop - West
T1006,C002,Webcam,85.0,2024-01-17,South,Webcam - South
T1007,C005,Headphones,120.0,2024-01-18,North,Headphones - North
T1008,C003,Monitor,350.0,2024-01-18,East,Monitor - East
T1009,C001,Docking Station,150.0,2024-01-19,North,Docking Station - North
T1010,C004,Mouse Pad,15.0,2024-01-19,West,Mouse Pad - West


In [0]:
from pyspark.sql.functions import concat_ws

# The Safe Way: Null-skipping concatenation
sales_df.withColumn(
    "full_description",
    concat_ws(" - ", col("product"), col("region")) # The first argument is ALWAYS the separator
).display()

transaction_id,customer_id,product,amount,transaction_date,region,full_description
T1001,C001,Laptop,1200.5,2024-01-15,North,Laptop - North
T1002,C002,Mouse,25.0,2024-01-15,South,Mouse - South
T1003,C003,Monitor,350.0,2024-01-16,East,Monitor - East
T1004,C001,Keyboard,45.0,2024-01-16,North,Keyboard - North
T1005,C004,Laptop,1150.0,2024-01-17,West,Laptop - West
T1006,C002,Webcam,85.0,2024-01-17,South,Webcam - South
T1007,C005,Headphones,120.0,2024-01-18,North,Headphones - North
T1008,C003,Monitor,350.0,2024-01-18,East,Monitor - East
T1009,C001,Docking Station,150.0,2024-01-19,North,Docking Station - North
T1010,C004,Mouse Pad,15.0,2024-01-19,West,Mouse Pad - West


In [0]:
from pyspark.sql.functions import col, concat_ws

# The Modern and Production Way
# We pass a Python Dictionary to do casting, math, and string logic all at once!

sales_df_clean = sales_df.withColumns({
    "amount_int": col("amount").cast("integer"),
    "tax_amount": col("amount") * 0.15,
    "description": concat_ws(" - ", col("product"), col("region"))
})

sales_df_clean.display()

transaction_id,customer_id,product,amount,transaction_date,region,amount_int,tax_amount,description
T1001,C001,Laptop,1200.5,2024-01-15,North,1200,180.075,Laptop - North
T1002,C002,Mouse,25.0,2024-01-15,South,25,3.75,Mouse - South
T1003,C003,Monitor,350.0,2024-01-16,East,350,52.5,Monitor - East
T1004,C001,Keyboard,45.0,2024-01-16,North,45,6.75,Keyboard - North
T1005,C004,Laptop,1150.0,2024-01-17,West,1150,172.5,Laptop - West
T1006,C002,Webcam,85.0,2024-01-17,South,85,12.75,Webcam - South
T1007,C005,Headphones,120.0,2024-01-18,North,120,18.0,Headphones - North
T1008,C003,Monitor,350.0,2024-01-18,East,350,52.5,Monitor - East
T1009,C001,Docking Station,150.0,2024-01-19,North,150,22.5,Docking Station - North
T1010,C004,Mouse Pad,15.0,2024-01-19,West,15,2.25,Mouse Pad - West


In [0]:
from pyspark.sql.functions import col, when

# Creating a dynamic column based on a condition
dynamic_df = sales_df.withColumns({
    "customer_segment": when(col("amount") > 1000, "Premium").otherwise("Standard")
})

dynamic_df.display()

transaction_id,customer_id,product,amount,transaction_date,region,customer_segment
T1001,C001,Laptop,1200.5,2024-01-15,North,Premium
T1002,C002,Mouse,25.0,2024-01-15,South,Standard
T1003,C003,Monitor,350.0,2024-01-16,East,Standard
T1004,C001,Keyboard,45.0,2024-01-16,North,Standard
T1005,C004,Laptop,1150.0,2024-01-17,West,Premium
T1006,C002,Webcam,85.0,2024-01-17,South,Standard
T1007,C005,Headphones,120.0,2024-01-18,North,Standard
T1008,C003,Monitor,350.0,2024-01-18,East,Standard
T1009,C001,Docking Station,150.0,2024-01-19,North,Standard
T1010,C004,Mouse Pad,15.0,2024-01-19,West,Standard


In [0]:
# Chaining multiple conditions for tiered logic
tiered_df = sales_df.withColumns({
    "value_tier": when(col("amount") > 1000, "High Value")
                 .when(col("amount") >= 100, "Mid Value")
                 .otherwise("Low Value")
})

tiered_df.display()

transaction_id,customer_id,product,amount,transaction_date,region,value_tier
T1001,C001,Laptop,1200.5,2024-01-15,North,High Value
T1002,C002,Mouse,25.0,2024-01-15,South,Low Value
T1003,C003,Monitor,350.0,2024-01-16,East,Mid Value
T1004,C001,Keyboard,45.0,2024-01-16,North,Low Value
T1005,C004,Laptop,1150.0,2024-01-17,West,High Value
T1006,C002,Webcam,85.0,2024-01-17,South,Low Value
T1007,C005,Headphones,120.0,2024-01-18,North,Mid Value
T1008,C003,Monitor,350.0,2024-01-18,East,Mid Value
T1009,C001,Docking Station,150.0,2024-01-19,North,Mid Value
T1010,C004,Mouse Pad,15.0,2024-01-19,West,Low Value


In [0]:
# The Danger of omitting .otherwise()
sales_df.withColumns({
    "is_laptop": when(col("product") == "Laptop", "Yes")
    # We forgot to add .otherwise("No")!
}).display()

transaction_id,customer_id,product,amount,transaction_date,region,is_laptop
T1001,C001,Laptop,1200.5,2024-01-15,North,Yes
T1002,C002,Mouse,25.0,2024-01-15,South,null
T1003,C003,Monitor,350.0,2024-01-16,East,null
T1004,C001,Keyboard,45.0,2024-01-16,North,null
T1005,C004,Laptop,1150.0,2024-01-17,West,Yes
T1006,C002,Webcam,85.0,2024-01-17,South,null
T1007,C005,Headphones,120.0,2024-01-18,North,null
T1008,C003,Monitor,350.0,2024-01-18,East,null
T1009,C001,Docking Station,150.0,2024-01-19,North,null
T1010,C004,Mouse Pad,15.0,2024-01-19,West,null


In [0]:
# Create a GroupedData object
grouped_df = sales_df.groupBy("region")

print(grouped_df)

GroupedData[grouping expressions: [region], value: [transaction_id: string, customer_id: string, product: string, amount: double, transaction_date: date, region: string], type: GroupBy]


In [0]:
# Simple Count Action
sales_df.groupBy("region").count().display()

region,count
North,7
East,3
South,3
null,1
West,3


In [0]:
# Total revenue per product
sales_df.groupBy("product").sum("amount").display()

product,sum(amount)
Webcam,85.0
Mouse Pad,15.0
Mouse,50.0
Monitor,1040.0
Keyboard,90.0
External Drive,80.0
Laptop,4701.0
Headphones,120.0
Unknown Product,null
Docking Station,150.0


In [0]:
# THE GOLDEN RULE: Filter before you group!
from pyspark.sql.functions import col

sales_df.filter(col("product").isin(["Laptop", "Monitor"])) \
        .groupBy("product") \
        .sum("amount") \
        .display()

product,sum(amount)
Monitor,1040.0
Laptop,4701.0


In [0]:
from pyspark.sql.functions import col, sum, avg, max, count, round

# The Professional Aggregation Pattern
report_df = sales_df.groupBy("region").agg(
    count("*").alias("total_transactions"),
    sum("amount").alias("total_revenue"),
    avg("amount").alias("avg_deal_size"),
    max("amount").alias("biggest_deal")
)

display(report_df)

region,total_transactions,total_revenue,avg_deal_size,biggest_deal
North,7,1640.5,273.4166666666667,1200.5
East,3,1850.0,616.6666666666666,1150.0
South,3,1310.5,436.8333333333333,1200.5
null,1,25.0,25.0,25.0
West,3,1505.0,501.6666666666667,1150.0


In [0]:
# Cleaning up the messy decimals
report_clean_df = sales_df.groupBy("region").agg(
    count("*").alias("total_transactions"),
    round(sum("amount"), 2).alias("total_revenue"),
    round(avg("amount"), 2).alias("avg_deal_size"),
    max("amount").alias("biggest_deal")
)

display(report_clean_df)

region,total_transactions,total_revenue,avg_deal_size,biggest_deal
North,7,1640.5,273.42,1200.5
East,3,1850.0,616.67,1150.0
South,3,1310.5,436.83,1200.5
null,1,25.0,25.0,25.0
West,3,1505.0,501.67,1150.0


In [0]:
# Multidimensional grouping (Drilling down)
granular_report_df = sales_df.groupBy("region", "product").agg(
    sum("amount").alias("revenue"),
    count("*").alias("sales_count")
).orderBy("region", "product")

display(granular_report_df)

region,product,revenue,sales_count
null,Mouse,25.0,1
East,Laptop,1150.0,1
East,Monitor,700.0,2
North,Docking Station,150.0,1
North,External Drive,80.0,1
North,Headphones,120.0,1
North,Keyboard,90.0,2
North,Laptop,1200.5,1
North,Unknown Product,null,1
South,Laptop,1200.5,1


In [0]:
# Register the DataFrame as a SQL View
sales_df.createOrReplaceTempView("sales_view")

In [0]:
# Writing SQL inside Python
sql_df = spark.sql("""
    SELECT 
        region, 
        product, 
        SUM(amount) as total_sales
    FROM sales_view
    WHERE amount > 100
    GROUP BY region, product
    ORDER BY total_sales DESC
""")

display(sql_df)

region,product,total_sales
South,Laptop,1200.5
North,Laptop,1200.5
West,Laptop,1150.0
East,Laptop,1150.0
East,Monitor,700.0
West,Monitor,340.0
North,Docking Station,150.0
North,Headphones,120.0


In [0]:
%sql
/* This is a pure SQL environment now */
SELECT 
    region, 
    COUNT(*) as transaction_count
FROM sales_view
GROUP BY region
ORDER BY transaction_count DESC

region,transaction_count
North,7
East,3
South,3
West,3
null,1


In [0]:
# Capture the output of the previous SQL cell
my_clean_df = _sqldf

# Now we can continue processing in Python
from pyspark.sql.functions import col

final_df = my_clean_df.filter(col("transaction_count") > 1)
display(final_df)

region,transaction_count
North,7
East,3
South,3
West,3


In [0]:
# Ask Spark to show us its execution plan
my_clean_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonSort [transaction_count#13815L DESC NULLS LAST]
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#9625]
               +- PhotonShuffleExchangeSink rangepartitioning(transaction_count#13815L DESC NULLS LAST, 16)
                  +- PhotonGroupingAgg(keys=[region#13519], functions=[finalmerge_count(merge count#13838L) AS count(1)#13836L])
                     +- PhotonShuffleExchangeSource
                        +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#9619]
                           +- PhotonShuffleExchangeSink hashpartitioning(region#13519, 16)
                              +- PhotonGroupingAgg(keys=[region#13519], functions=[partial_count(1) AS count#13838L])
                                 +- PhotonRowToColumnar
                                    +- FileScan csv [region#13519] B

In [0]:
# Define our target destination
output_path = "/Volumes/workspace/default/course_data/output/sales_report"

# Basic Write Command
report_df.write.format("delta").mode("overwrite").save(output_path)

In [0]:
# Read the data back to verify
df_verify = spark.read.format("delta").load(output_path)
display(df_verify)

region,total_transactions,total_revenue,avg_deal_size,biggest_deal
North,7,1640.5,273.4166666666667,1200.5
East,3,1850.0,616.6666666666666,1150.0
South,3,1310.5,436.8333333333333,1200.5
null,1,25.0,25.0,25.0
West,3,1505.0,501.6666666666667,1150.0


In [0]:
# Save as a Managed Table in the Catalog
table_name = "workspace.default.sales_report_managed"

report_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

In [0]:
%sql
-- Optimize the table and cluster the data by region
OPTIMIZE workspace.default.sales_report_managed
ZORDER BY (region)

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1732), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1773937350959, 1773937351699, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
# Simulate a disastrous accidental overwrite
table_name = "workspace.default.sales_report_managed"

# We grab just 1 row and completely overwrite the production table!
report_df.limit(1).write.format("delta").mode("overwrite").saveAsTable(table_name)

In [0]:
%sql
SELECT * FROM workspace.default.sales_report_managed

region,total_transactions,total_revenue,avg_deal_size,biggest_deal
North,7,1640.5,273.4166666666667,1200.5


In [0]:
%sql
-- View the transaction log history
DESCRIBE HISTORY workspace.default.sales_report_managed

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-03-19T16:23:12.000Z,73530545840616,thulani_bw@outlook.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(836986748628288),01814221-f83f-4eae-9a76-d18e7327737f,0319-161446-jd2cytg2-v2n,3,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1732, numDeletionVectorsRemoved -> 0, numOutputRows -> 1, numOutputBytes -> 1560)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
3,2026-03-19T16:22:30.000Z,73530545840616,thulani_bw@outlook.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(836986748628288),13726877-5e62-4b49-a7cf-488cc4b628fb,0319-161446-jd2cytg2-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1732, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1732)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
2,2026-03-19T15:43:11.000Z,73530545840616,thulani_bw@outlook.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(836986748628288),3240ef0d-4266-413e-b89c-aefd345f4f20,0319-151620-8n0o8817-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1732, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1732)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-19T15:41:29.000Z,73530545840616,thulani_bw@outlook.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(836986748628288),1005a90a-f1c4-4eef-89a1-1c15f2fd9d36,0319-151620-8n0o8817-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1732, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1732)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-19T15:38:31.000Z,73530545840616,thulani_bw@outlook.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(836986748628288),5cce2411-3c5d-4ca5-ab77-590dc178380c,0319-151620-8n0o8817-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 1732)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


In [0]:
%sql
-- Restore the table to its very first version
RESTORE TABLE workspace.default.sales_report_managed TO VERSION AS OF 0

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
1732,1,1,1,1560,1732


In [0]:
%sql
SELECT * FROM workspace.default.sales_report_managed

region,total_transactions,total_revenue,avg_deal_size,biggest_deal
North,7,1640.5,273.4166666666667,1200.5
East,3,1850.0,616.6666666666666,1150.0
South,3,1310.5,436.8333333333333,1200.5
null,1,25.0,25.0,25.0
West,3,1505.0,501.6666666666667,1150.0
